In [0]:
# ============================================================
# BANKING LAKEHOUSE - IMPORTS AND CONFIGURATION
# ============================================================

from src.config.settings import (
    PIPELINE_NAME,
    BRONZE_TABLE,
    SILVER_TABLE,
    QUARANTINE_TABLE,
    GOLD_TABLE,
    GOLD_TRANSACTION_CONTRIBUTION_TABLE,
    GOLD_CONTROL_TABLE,
    DQ_TABLE,
    CONTROL_TABLE,
    FORCE_DQ_RETRY_FAILURE
)

from src.ingestion.sample_source import create_sample_transactions

from src.bronze.bronze_processor import ingest_to_bronze

from src.silver.silver_processor import process_silver

from src.quality.data_quality import reconcile_data_quality

from src.gold.gold_processor import process_gold

from src.utilities.batch_control import (
    is_batch_successful,
    record_batch_status
)

from pyspark.sql import functions as F


print(f"Pipeline initialized: {PIPELINE_NAME}")

# Development table overrides

BRONZE_TABLE = "workspace.default.dev_bronze_transactions"
SILVER_TABLE = "workspace.default.dev_silver_transactions"
QUARANTINE_TABLE = "workspace.default.dev_quarantine_transactions"

GOLD_TABLE = "workspace.default.dev_gold_customer_metrics"

GOLD_TRANSACTION_CONTRIBUTION_TABLE = (
    "workspace.default.dev_gold_transaction_contributions"
)

GOLD_CONTROL_TABLE = "workspace.default.dev_gold_batch_control"
DQ_TABLE = "workspace.default.dev_etl_dq_summary"
CONTROL_TABLE = "workspace.default.dev_etl_batch_control"

print("Development configuration loaded.")

In [0]:
# ============================================================
# MAIN PIPELINE ORCHESTRATION
# ============================================================

def run_pipeline(batch_id, incoming_df=None):

    print("=" * 65)
    print(f"PIPELINE: {PIPELINE_NAME}")
    print(f"BATCH ID: {batch_id}")
    print("=" * 65)

    try:

        # 1. Batch idempotency check
        if is_batch_successful(spark, batch_id, CONTROL_TABLE):

            print(f"Batch {batch_id} already completed.")
            print("Skipping execution.")

            return "SKIPPED"

        # 2. Source ingestion
        print("\nSTAGE 1: SOURCE INGESTION")

        if incoming_df is None:
            incoming_df = create_sample_transactions(spark)

        # 3. Bronze ingestion
        print("\nSTAGE 2: BRONZE INGESTION")

        bronze_current_batch = ingest_to_bronze(
            spark,
            incoming_df,
            batch_id,
            BRONZE_TABLE
        )

        # 4. Silver processing
        print("\nSTAGE 3: SILVER PROCESSING")

        silver_result = process_silver(
            spark,
            bronze_current_batch,
            SILVER_TABLE,
            QUARANTINE_TABLE
        )

        silver_valid_batch = silver_result["silver_valid_batch"]

        bronze_count = silver_result["bronze_count"]
        valid_count = silver_result["valid_count"]
        quarantine_count = silver_result["quarantine_count"]
        duplicate_count = silver_result["duplicate_count"]

        invalid_transaction_versions = silver_result[
            "invalid_transaction_versions"
        ]

        # 5. Data quality reconciliation
        print("\nSTAGE 4: DATA QUALITY")

        reconcile_data_quality(
            spark,
            batch_id,
            bronze_count,
            valid_count,
            quarantine_count,
            duplicate_count,
            DQ_TABLE,
            force_failure=FORCE_DQ_RETRY_FAILURE
        )

        # 6. Gold processing
        print("\nSTAGE 5: GOLD PROCESSING")

        process_gold(
            spark,
            batch_id,
            silver_valid_batch,
            GOLD_TABLE,
            GOLD_TRANSACTION_CONTRIBUTION_TABLE,
            GOLD_CONTROL_TABLE,
            invalid_transaction_versions=invalid_transaction_versions
        )

        # 7. Record successful batch
        record_batch_status(
            spark,
            batch_id,
            "SUCCESS",
            CONTROL_TABLE
        )

        print("\nPIPELINE COMPLETED SUCCESSFULLY")

        return "SUCCESS"

    except Exception as e:

        print(f"\nPIPELINE FAILED: {str(e)}")

        record_batch_status(
            spark,
            batch_id,
            "FAILED",
            CONTROL_TABLE,
            error_message=str(e)[:2000]
        )

        raise

In [0]:
batch_id = "dev_integration_20261105_01"

incoming_df = create_sample_transactions(spark)

result = run_pipeline(batch_id, incoming_df=incoming_df)

print("Pipeline result:", result)

In [0]:
import importlib
import src.silver.silver_processor as silver_module

importlib.reload(silver_module)

process_silver = silver_module.process_silver

print("Silver processor refreshed.")

In [0]:
# ============================================================
# PIPELINE EXECUTION SUMMARY
# ============================================================

batch_summary = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("processed_at").desc())
)

display(batch_summary)